In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
import sys
import time
warnings.filterwarnings('ignore')

from tqdm import tqdm
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image

In [2]:
try:
    import google.colab
    from google.colab import drive
    !uv pip install anomalib
    !uv pip install open-clip-torch
    drive.mount('/content/drive', force_remount=True)
    PROJECT_ROOT = Path('/content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation') # 본인 경로 수정: Mac/Window
except ImportError:
    PROJECT_ROOT = Path.cwd().parents[1]

os.chdir(PROJECT_ROOT) # 현재 경로 수정
print(f"Current working directory: {os.getcwd()}")

Using Python 3.12.12 environment at: /usr
Audited 1 package in 103ms
Using Python 3.12.12 environment at: /usr
Audited 1 package in 96ms
Mounted at /content/drive
Current working directory: /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation


In [3]:
# 1. 68GB를 먹고 있는 유령 프로세스를 강제로 죽입니다.
!fuser -k /dev/nvidia*

# 2. 런타임이 끊겼다 다시 붙으면 아래를 확인하세요.
import torch
print(f"현재 GPU 사용량: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
# 이 값이 0.00 GB라면 이제 예전처럼 6개 클래스 완주가 가능해집니다.
free, total = torch.cuda.mem_get_info()
print(f"진짜 빈 공간: {free / 1024**3:.2f} GB")

현재 GPU 사용량: 0.00 GB
진짜 빈 공간: 78.90 GB


In [3]:
# !pip install wandb -q
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: tonyntrish60 (tonyntrish60-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os

# 1. WandB를 오프라인 모드로 설정 (네트워크 전송 대기 시간 제거)
os.environ["WANDB_MODE"] = "offline"

# 2. 결과 저장 경로를 구글 드라이브가 아닌 코랩 로컬로 설정
# (구글 드라이브 병목을 피하기 위함입니다)
LOCAL_RESULT_PATH = "/content/results"
if not os.path.exists(LOCAL_RESULT_PATH):
    os.makedirs(LOCAL_RESULT_PATH)

print("WandB Offline 모드 활성화 및 로컬 저장 경로 설정 완료!")

WandB Offline 모드 활성화 및 로컬 저장 경로 설정 완료!


In [5]:
# 1. 기존 학습 중단 (Stop 버튼 클릭) 후 실행
# 2. 구글 드라이브의 데이터셋을 코랩 로컬로 압축 해제 (가장 빠름)
# 만약 .zip 파일이 있다면 unzip을 쓰고, 폴더채로 옮기려면 cp를 씁니다.

import os

# 드라이브 경로 (사용자님 경로)
DRIVE_DATA_PATH = "/content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/dataset/MMAD/GoodsAD"
# 로컬 복사 경로
LOCAL_DATA_PATH = "/content/dataset/GoodsAD"

if not os.path.exists(LOCAL_DATA_PATH):
    print("데이터 복사 시작 (이 작업은 한 번만 하면 됩니다)...")
    !mkdir -p /content/dataset
    # 폴더 통째로 복사 (이미지가 많으면 5~10분 소요될 수 있지만, 이후 학습은 광속입니다)
    !cp -r "{DRIVE_DATA_PATH}" /content/dataset/
    print("복사 완료!")

# 3. runtime.yaml의 data root 경로를 '/content/dataset'으로 수정 후 다시 실행

데이터 복사 시작 (이 작업은 한 번만 하면 됩니다)...
복사 완료!


In [ ]:
import cv2
import os
from pathlib import Path
from tqdm import tqdm

def resize_all_images(root_path, size=(256, 256)):
    image_paths = list(Path(root_path).rglob("*.png")) + list(Path(root_path).rglob("*.jpg"))
    print(f"총 {len(image_paths)}개 이미지 리사이즈 시작...")

    for p in tqdm(image_paths):
        img = cv2.imread(str(p))
        if img is not None:
            # 비율 무시하고 강제로 size로 고정 (이게 에러 해결의 핵심)
            resized = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(p), resized)

# 로컬에 복사한 데이터셋 경로로 실행
resize_all_images("/content/dataset/GoodsAD")

In [6]:
import os
import torch
import gc
from pathlib import Path

# 1. 끈질긴 유령 프로세스 강제 종료 (GPU 메모리 0GB로 초기화)
# !fuser -k /dev/nvidia*

# 2. 메모리 최적화 환경 변수 설정 (반드시 맨 처음에!)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128,expandable_segments:True"
torch.set_float32_matmul_precision('medium')

# 3. 메모리 청소기 가동
gc.collect()
torch.cuda.empty_cache()

from scripts.train_anomalib import Anomalibs

if __name__ == "__main__":
    # 실제 드라이브 경로
    real_path = "/content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/dataset/MMAD"

    app = Anomalibs(config_path="configs/runtime.yaml")

    # 경로 강제 덮어쓰기
    app.data_root = Path(real_path)
    app.loader.root = Path(real_path)

    print(f"--- 경로 강제 설정 완료 ---")
    print(f"Target Path: {app.data_root / 'GoodsAD'}")
    print(f"Exists?: {(app.data_root / 'GoodsAD').exists()}")

    # 카테고리 확인
    cats = app.loader.get_categories("GoodsAD")
    print(f"Found Categories: {cats}")

    if cats:
        # GPU 잔여 메모리 최종 확인 (0에 가까워야 함)
        free_mem = torch.cuda.mem_get_info()[0] / 1024**3
        print(f"학습 시작 전 GPU 여유 공간: {free_mem:.2f} GB")

        app.fit_all()

TrainAnomalib - INFO - Initialized - model: efficientad, device: cuda
TrainAnomalib - INFO - [1/6] GoodsAD/cigarette_box
TrainAnomalib - INFO - Fitting efficientad - GoodsAD/cigarette_box


Device: NVIDIA A100-SXM4-80GB
--- 경로 강제 설정 완료 ---
Target Path: /content/drive/Othercomputers/my_notebook/lion_final_pro_multimodal-anomaly-report-generation/dataset/MMAD/GoodsAD
Exists?: True
Found Categories: ['cigarette_box', 'drink_bottle', 'drink_can', 'food_bottle', 'food_box', 'food_package']
학습 시작 전 GPU 여유 공간: 78.81 GB


INFO:lightning_fabric.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:lightning_fabric.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor     │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor    │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator        │      0 │ train │     0 │
│ 3 │ model          │ EfficientAdModel │ 20.7 M │ train │     0 │
└───┴────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 20.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 20.7 M                                                                                               
Total estimated model params size (MB): 82                                                                         
Modules in train mode: 50                                                                                          
Modules in eval mode: 9                                                                                            
Total FLOPs: 0

...

INFO:lightning_fabric.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.
TrainAnomalib - INFO - Fitting GoodsAD/cigarette_box done and memory cleared
TrainAnomalib - INFO - [2/6] GoodsAD/drink_bottle
TrainAnomalib - INFO - Fitting efficientad - GoodsAD/drink_bottle
INFO:lightning_fabric.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:lightning_fabric.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:lightning_fabric.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor     │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor    │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator        │      0 │ train │     0 │
│ 3 │ model          │ EfficientAdModel │ 20.7 M │ train │     0 │
└───┴────────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 20.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 20.7 M                                                                                               
Total estimated model params size (MB): 82                                                                         
Modules in train mode: 50                                                                                          
Modules in eval mode: 9                                                                                            
Total FLOPs: 0

Calculate Validation Dataset Quantiles: 100%|██████████| 146/146 [03:23<00:00,  1.40s/it]
TrainAnomalib - ERROR - Error during fitting: Sizes of tensors must match except in dimension 0. Expected size 3000 but got size 3024 for tensor number 2 in the list.


RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 3000 but got size 3024 for tensor number 2 in the list.

In [ ]:
from scripts.train_anomalib import Anomalibs

if __name__ == "__main__":
    app = Anomalibs(config_path="configs/runtime.yaml")
    # 학습된 모델을 불러와 모든 카테고리 예측 및 JSON 저장
    app.predict_all(save_json=True)

In [ ]:
from scripts.VIS_RAG import VisualRAG

if __name__ == "__main__":
    # 예측 단계에서 생성된 JSON 결과가 필요합니다.
    rag = VisualRAG()
    rag.run()